# Semantic Cache Quickstart

A **cache HIT** means the cache reuses a previously stored answer for a similar query; a **cache MISS** means the query is too different, so the model is called instead. Every lookup, vector search, scoring, and HIT/MISS decision is owned by the `MLCache` instance -- this notebook builds one, queries it directly, and then wraps it around a real LLM with `CachedLLM`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from mlcache import CacheEntry, CacheKey, CacheLookup, MLCache, Query, Response
from mlcache.embeddings import HashingEmbeddingProvider

embedder = HashingEmbeddingProvider(dimensions=64)


def surviving_cache_keys(cache: MLCache) -> list[str]:
    """Cache keys the eviction policy still tracks, sorted."""
    policy = cache.runtime.gateway.eviction_policy
    if policy is None:
        return []
    return sorted(str(key) for key in policy.tracked_keys())

## Build the cache

`MLCache.from_preset` is the normal way to construct a working cache in one call:

- **`scorer`**: how candidate reuse is scored -- `"cosine"` (plain similarity, used here) or `"ensemble"` (a trained combination of several scorers).
- **`threshold`**: the minimum score for a HIT.
- **`persistence`**: whether cache state is written to `root_dir` as JSON (`False` here = in-memory only).
- **`eviction_policy`** / **`max_entries`**: which entry to drop (`"lru"`, `"lfu"`, or `"fifo"`) once the cache exceeds `max_entries`.


In [2]:
cache = MLCache.from_preset(
    root_dir="/tmp/mlcache_quickstart",
    scorer="cosine",
    threshold=0.60,
    persistence=False,
    eviction_policy="lru",
    max_entries=1000,
)

print("scorer   :", cache.scorer.name)
print("threshold:", cache.threshold)
print("components:", cache.components)

scorer   : Cosine
threshold: 0.6
components: {'gateway': 'SemanticCacheGateway', 'oracle': 'TrainableSemanticCacheOracle', 'kv_store': 'InMemoryKVStore', 'vector_store': 'InMemoryVectorStore', 'feature_builder': 'NormalizedHadamardFeatureBuilder', 'scorer': 'CosineScorer:Cosine', 'shadow_collector': None, 'judge_training_store': None, 'query_level_policy': None, 'query_level_shadow_store': None, 'query_record_store': 'InMemoryQueryCalibrationRecordStore', 'audit_logger': None, 'metrics_sink': None, 'diagnostics_reporter': None}


## Run example queries

Index one entry, then look up a near-duplicate query (expected **HIT**) and an unrelated query (expected **MISS**).

In [3]:
cache.put(
    CacheEntry(
        cache_key=CacheKey("semantic-caching-explainer"),
        query=Query("What is semantic caching?"),
        response=Response("Semantic caching reuses a previous response when a new query is semantically close enough."),
        embedding=embedder.embed("What is semantic caching?"),
    )
)


def lookup(query: str) -> None:
    result = cache.lookup_with_decision(CacheLookup(query=Query(query), embedding=embedder.embed(query)))
    decision = result.decision
    print(f"query    : {query}")
    print(f"status   : {decision.status}")
    print(f"score    : {decision.score}")
    print(f"threshold: {decision.threshold}")
    print(f"answer   : {result.response}")
    print()


lookup("What is semantic caching, exactly?")  # near-duplicate -> HIT
lookup("What's the weather like today?")   # unrelated -> MISS

query    : What is semantic caching, exactly?
status   : hit
score    : 0.670820393249937
threshold: 0.6
answer   : Semantic caching reuses a previous response when a new query is semantically close enough.

query    : What's the weather like today?
status   : miss
score    : 0.223606797749979
threshold: 0.6
answer   : None



## Configurable eviction policies

When a cache is bounded with `max_entries`, an **eviction policy** decides which entry to drop once that bound is exceeded. This is independent of the semantic HIT/MISS decision above -- the scorer still owns reuse; the eviction policy only governs which *already admitted* entry is removed to make room. `MLCache.from_preset(..., eviction_policy=..., max_entries=...)` selects one of three policies:

- **`lru`** -- evict the least *recently used* entry.
- **`lfu`** -- evict the least *frequently used* entry.
- **`fifo`** -- evict the oldest *inserted* entry.

Each cache entry tracks `created_at`/insertion order, `last_accessed`, and `access_count`; a cache HIT refreshes `last_accessed` and bumps `access_count`. The cell below builds a small `max_entries=2` cache per policy and replays the same access pattern -- insert `alpha`, insert `beta`, HIT `alpha`, insert `gamma` -- to show which two entries survive (see `scripts/run_eviction_policy_example.py` for a standalone version).

In [15]:
_items = {
    "alpha": "Alpha is the first response.",
    "beta": "Beta is the second response.",
    "gamma": "Gamma is the third response.",
}


def _entry(key: str) -> CacheEntry:
    return CacheEntry(
        cache_key=CacheKey(key),
        query=Query(key),
        response=Response(_items[key]),
        embedding=embedder.embed(key),
    )


def _run_eviction_demo(policy: str) -> list[str]:
    eviction_cache = MLCache.from_preset(
        root_dir=f"/tmp/mlcache_quickstart_evict_{policy}",
        scorer="cosine",
        threshold=0.99,        # exact-embedding match -> HIT
        persistence=False,
        eviction_policy=policy,  # "lru" | "lfu" | "fifo"
        max_entries=2,            # capacity bound that triggers eviction
    )
    eviction_cache.put(_entry("alpha"))
    eviction_cache.put(_entry("beta"))
    eviction_cache.lookup_with_decision(
        CacheLookup(query=Query("alpha"), embedding=embedder.embed("alpha"))
    )  # HIT on alpha -> most recently *and* most frequently used
    eviction_cache.put(_entry("gamma"))  # exceeds capacity -> one eviction
    return surviving_cache_keys(eviction_cache)


print("Access pattern: put alpha, put beta, HIT alpha, put gamma (max_entries=2)\n")
for policy in ("fifo", "lru", "lfu"):
    print(f"{policy.upper():5s} survivors={_run_eviction_demo(policy)}")

Access pattern: put alpha, put beta, HIT alpha, put gamma (max_entries=2)

FIFO  survivors=['beta', 'gamma']
LRU   survivors=['alpha', 'gamma']
LFU   survivors=['alpha', 'gamma']


## Plugging a real LLM into the cache

The `cache` built above is empty and ready to serve. `CachedLLM` adds the missing piece: cache-first orchestration around any model that implements `LLMClient.generate(prompt) -> LLMResponse`.

- **HIT** -> return the cached answer, the LLM is never called.
- **MISS** -> call the LLM, then write the new answer into the cache for next time.

`OpenAICompatibleLLM` implements `LLMClient` for *any* OpenAI-compatible `/v1/chat/completions` server -- vLLM, the real OpenAI API, Azure OpenAI, Together, etc. Only `base_url` / `api_key` / `model` change between backends; the cache wiring is identical.

In [40]:
# A vLLM server was started locally from the HF cache (no network/model download):
#   LD_LIBRARY_PATH=.conda/lib HF_HUB_OFFLINE=1 .conda/bin/python -m vllm.entrypoints.openai.api_server \
#       --model Qwen/Qwen3-8B-AWQ --port 8000 --api-key local-vllm-token --max-model-len 8192
from mlcache.llm_wrapper import CachedLLM
from mlcache.llm_providers import DEFAULT_ASSISTANT_SYSTEM_PROMPT, OpenAICompatibleLLM

llm = OpenAICompatibleLLM(
    base_url="http://localhost:8000/v1",
    api_key="local-vllm-token",
    model="Qwen/Qwen3-8B-AWQ",
    system_prompt=DEFAULT_ASSISTANT_SYSTEM_PROMPT + " /no_think",
    default_max_tokens=200,
)

# Reuse the cache built above -- same scorer, threshold, and eviction policy.
cached_llm = CachedLLM(llm=llm, cache=cache)

In [42]:
# This prompt is new -> MISS -> the real Qwen3-8B-AWQ model is called,
# and its answer is written into the cache.
prompt = "i want you to explain to me semantic caching in one sentence, please."

result1 = cached_llm.generate(prompt)
print("source   :", result1.source)
print("cache_key:", result1.cache_key)
print("answer   :", result1.text)

source   : cache
cache_key: 2e8fa0dc30ee783b
answer   : <think>

</think>

Semantic caching is a technique where frequently accessed or computed data, based on meaning or context, is stored to improve efficiency and reduce redundant processing.


In [ ]:
# Same prompt again: now a HIT -> served straight from the cache, the LLM is
# NOT called this time.
result2 = cached_llm.generate(prompt)
print("source   :", result2.source)
print("score    :", result2.score, " threshold:", result2.threshold)
print("answer   :", result2.text)

assert result2.source == "cache" and result2.text == result1.text

source   : cache
score    : 1.0  threshold: 0.6
answer   : <think>

</think>

Semantic caching is a technique where frequently accessed or computed data, based on meaning or context, is stored to improve efficiency and reduce redundant processing.


### Swapping in a different LLM API

Because `CachedLLM` only depends on the small `LLMClient.generate()` protocol, pointing it at a different model or provider is a one-line change to `OpenAICompatibleLLM` -- the cache, scorer, and threshold are untouched:

```python
# e.g. the real OpenAI API instead of the local vLLM server
openai_llm = OpenAICompatibleLLM(
    base_url="https://api.openai.com/v1",
    api_key="sk-...",
    model="gpt-4o-mini",
)
cached_llm_openai = CachedLLM(llm=openai_llm, cache=cache)
cached_llm_openai.generate("...")  # same cache, different backend
```

Any backend that speaks the OpenAI `/v1/chat/completions` protocol (vLLM, OpenAI, Azure OpenAI, Together, Fireworks, ...) works this way -- only `base_url` / `api_key` / `model` change.













### Automated online training and calibration


Everything above used a fixed `threshold=0.60`. The real system can also keep itself tuned automatically while serving traffic:

1. On each lookup, a `SemanticReuseJudge` labels the top-k retrieved candidates as REUSABLE (H1) or NOT_REUSABLE (H0) and a `SplitJudgeTrainingStore` accumulates them into train/calibration buckets.
2. After each lookup, `ConservativeRefitPolicy` checks those counts (plus FPR/TPR monitors and cooldowns) and decides whether to recalibrate the threshold or refit the scorer.
3. If so, the oracle refits/recalibrates on a background thread and atomically swaps in the new scorer and threshold -- serving continues uninterrupted.

To enable this, pass a `judge=` to `from_preset` along with `config=MLCacheRuntimeConfig(shadow=ShadowRuntimeConfig(enabled=True), refit=RuntimeRefitConfig(auto_refit=True))`. No extra calls are needed -- `cached_llm.generate(...)` triggers it as a side effect of normal lookups, once enough H0/H1 examples have been collected.

### The threshold moves on its own as traffic arrives

The "Automated online training and calibration" section above described the loop; here it actually runs. This cache is built with a `judge=` and `shadow.enabled=True` / `refit.auto_refit=True`, so every `lookup_with_decision` call judges the retrieved candidates as REUSABLE (H1) or NOT_REUSABLE (H0) and lets `ConservativeRefitPolicy` decide whether to recalibrate -- no manual `prefit_and_calibrate` call.

The judge below is a tiny stand-in for a real one: it labels a pair REUSABLE if the query and candidate are about the same topic. The gate thresholds (`min_calibration_h0`, `min_decisions_between_calibrations`, ...) are lowered from their production defaults purely so recalibration is visible after a handful of prompts instead of hundreds.

Three topics ("caching", "password reset", "capital of France") are indexed as anchors. Each new prompt below is a near-duplicate of one anchor's wording but about a *different* topic -- a textbook hard negative: the cosine scorer thinks it's a likely match, but the judge says NOT_REUSABLE. Watch `cache.threshold` rise to push those hard negatives below the bar.

In [ ]:
from mlcache import (
    JudgeDecision,
    JudgeLabel,
    JudgeRequest,
    JudgeResult,
    MLCacheRuntimeConfig,
    RuntimeRefitConfig,
    SemanticReuseJudge,
    ShadowRuntimeConfig,
    ConservativeRefitConfig,
    ConservativeRefitPolicy,
)

# Ground-truth topic for each text -- stands in for a real judge's verdict.
TOPIC = {
    "What is semantic caching?": "caching",
    "How do I reset my password?": "password",
    "What is the capital of France?": "geo_france",
    "What's the weather like today?": "weather",
    "How can I reset my password?": "password",
    "What is semantic search?": "search",
    "What is the capital of Germany?": "geo_germany",
}


class TopicJudge(SemanticReuseJudge):
    """REUSABLE iff the query and candidate are about the same topic."""

    @property
    def name(self) -> str:
        return "topic-judge"

    def judge(self, request: JudgeRequest) -> JudgeResult:
        same_topic = TOPIC.get(str(request.query)) == TOPIC.get(str(request.candidate_query))
        label = JudgeLabel.REUSABLE if same_topic else JudgeLabel.NOT_REUSABLE
        return JudgeResult(request=request, decision=JudgeDecision(label=label))


auto_cache = MLCache.from_preset(
    root_dir="/tmp/mlcache_auto_threshold",
    scorer="cosine",
    threshold=0.5,
    persistence=False,
    judge=TopicJudge(),
    config=MLCacheRuntimeConfig(
        shadow=ShadowRuntimeConfig(enabled=True, top_k=3, calibration_every_n=1),
        refit=RuntimeRefitConfig(auto_refit=True, target_false_accept_rate=0.7),
    ),
)

# Lower the gate thresholds so recalibration is visible after a few prompts
# (production defaults wait for hundreds of judged examples).
auto_cache.runtime.oracle.refit_policy = ConservativeRefitPolicy(
    ConservativeRefitConfig(
        min_h0_for_calibration=1,
        min_new_h0_for_calibration=1,
        min_decisions_between_calibrations=1,
        max_decisions_between_calibrations=1,
        min_calibration_h0=1,
        min_calibration_h1=0,
        fpr_wilson_margin=1.0,
        min_new_h0_for_refit=10**9,
        min_new_h1_for_refit=10**9,
    )
)

for text, key in [
    ("What is semantic caching?", "anchor-caching"),
    ("How do I reset my password?", "anchor-password"),
    ("What is the capital of France?", "anchor-geo"),
]:
    auto_cache.put(CacheEntry(cache_key=CacheKey(key), query=Query(text), response=Response(f"answer: {text}"), embedding=embedder.embed(text)))

print("initial threshold:", auto_cache.threshold)

incoming_prompts = [
    "What's the weather like today?",
    "How can I reset my password?",
    "What is semantic search?",
    "What is the capital of Germany?",
]
for prompt in incoming_prompts:
    auto_cache.lookup_with_decision(CacheLookup(query=Query(prompt), embedding=embedder.embed(prompt)))
    refit = auto_cache.runtime.oracle.last_refit_decision
    print(f"after {prompt!r:35s} threshold={auto_cache.threshold}  ({refit.action}: {refit.reason})")

initial threshold: 0.5
after "What's the weather like today?"    threshold=0.5  (noop: threshold_recalibration_not_activated)
after 'How can I reset my password?'      threshold=0.5  (noop: threshold_recalibration_not_activated)
after 'What is semantic search?'          threshold=1.0  (recalibrate_threshold: fpr_above_limit)
after 'What is the capital of Germany?'   threshold=0.6666666865348816  (recalibrate_threshold: fpr_above_limit)
